In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os
import sys
from pathlib import Path

# Поменяй путь под свою папку на Google Drive.
# В этой папке должны лежать: dataset.py, model.py, trainer.py.
PROJECT_DIR = Path('/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training')
DATA_DIR = PROJECT_DIR / 'data' / 'gold_labeled_data'

os.chdir(PROJECT_DIR)

In [7]:
import gc
import json
import random
import warnings
from datetime import datetime
from typing import Optional
import re
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from jinja2 import Template
from tqdm.auto import tqdm


warnings.filterwarnings('ignore')

RANDOM_SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA memory, GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA memory, GB: 94.97


In [3]:
# !pip install -q bert-score

In [2]:
# %pip install --upgrade transformers

In [4]:
# # Для fresh Colab лучше выполнить эту ячейку один раз, затем Runtime -> Restart runtime.
# # Особенно важно удалить старый torchao: он может конфликтовать с peft/get_peft_model.

# !pip uninstall -y torchao
# !pip install -q -U transformers accelerate peft bitsandbytes datasets evaluate sentencepiece


# Грузим датасеты

In [8]:
FULL_DATA_XLSX = DATA_DIR / 'app_reviews_ru_gold_labeled.xlsx'
TRAIN_XLSX = DATA_DIR / 'train.xlsx'
TEST_XLSX = DATA_DIR / 'test.xlsx'

DROP_COLS = ['Unnamed: 0', 'Unnamed: 0.1', 'lable_silver_common_AI', 'summary_silver_GPT']


def load_review_data():
    if TRAIN_XLSX.exists() and TEST_XLSX.exists():
        print('Loading existing train.xlsx/test.xlsx')
        train_df = pd.read_excel(TRAIN_XLSX)
        test_df = pd.read_excel(TEST_XLSX)
    else:
        print('train.xlsx/test.xlsx were not found, creating stratified split from FINAL_GOLD')
        df = pd.read_excel(FULL_DATA_XLSX, sheet_name='FINAL_GOLD')
        df = df.drop(columns=DROP_COLS, errors='ignore')
        df = df.dropna(subset=['text', 'label_gold']).reset_index(drop=True)

        train_df, test_df = train_test_split(
            df,
            test_size=0.15,
            random_state=RANDOM_SEED,
            stratify=df['label_gold']
        )

        train_df = train_df.reset_index(drop=True)
        test_df = test_df.reset_index(drop=True)
        train_df.to_excel(TRAIN_XLSX, index=False)
        test_df.to_excel(TEST_XLSX, index=False)

    train_df = train_df.drop(columns=DROP_COLS, errors='ignore')
    test_df = test_df.drop(columns=DROP_COLS, errors='ignore')

    for df in [train_df, test_df]:
        df['text'] = df['text'].fillna('').astype(str)
        df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
        df['thumbs_up_count'] = pd.to_numeric(df['thumbs_up_count'], errors='coerce')

    return train_df, test_df

train_data, test_data = load_review_data()

print('Train:', train_data.shape)
print('Test:', test_data.shape)
print('Train columns:', train_data.columns.tolist())
print('Class distribution:')
print(train_data['label_gold'].value_counts(normalize=True).round(4))
train_data.head()

Loading existing train.xlsx/test.xlsx
Train: (2714, 6)
Test: (480, 6)
Train columns: ['app_name', 'text', 'rating', 'thumbs_up_count', 'label_gold', 'summary_gold']
Class distribution:
label_gold
user_experience    0.4749
bug_report         0.2598
rating             0.2119
feature_request    0.0534
Name: proportion, dtype: float64


,app_name,text,rating,thumbs_up_count,label_gold,summary_gold
0,Avito,программа Авито очень помогает в продаже б.у т...,5,0,user_experience,Положительный опыт использования приложения
1,VK,Скачала и пользовалась с 2015 года. Новый отзы...,1,24,user_experience,Недовольство модерацией и поддержкой
2,Yandex Maps,Сотрудники Яндекс Поддержки продались и сотруд...,1,5,user_experience,Мешает навязчивая реклама
3,Telegram,"прошу, сделайте так чтоб подарки можно было пр...",5,0,feature_request,Разрешить продажу подарков в любое время
4,Yandex Maps,настройте приложение опять заедает и вылетает,1,3,bug_report,Приложение вылетает


# System prompt

In [9]:
system_prompt = Template(
"""
Ты — NLP-ассистент для анализа русскоязычных отзывов на приложения Google Play.

Тебе будет передан один отзыв в формате:
text: текст отзыва
rating: оценка приложения от 1 до 5
thumbs_up_count: количество лайков отзыва

Твоя задача:
1. Определить основной класс отзыва строго из списка:
- bug_report
- feature_request
- user_experience
- rating

Правила разметки:
- bug_report: пользователь сообщает об ошибке, сбое, зависании, некорректной работе, потере данных.
- feature_request: пользователь просит добавить или изменить функцию. Если одновременно есть явная ошибка и просьба, выбери bug_report.
- user_experience: пользователь описывает опыт использования, удобство, неудобство, плюсы/минусы без явного бага и без просьбы новой функции.
- rating: короткая общая оценка без полезной конкретики.

Приоритет при смешанных случаях:
bug_report > feature_request > user_experience > rating

2. Сформировать summary:
- одна короткая фраза на русском;
- не более 12 слов;
- без выдуманных деталей;
- только главная мысль из отзыва.

Верни только валидный JSON без markdown и пояснений:
{"label": "bug_report|feature_request|user_experience|rating", "summary": "короткое summary"}

Примеры:
Отзыв:
text: все приложение не работает вообще, не заходит и не грузит
rating: 1
thumbs_up_count: 0
Ответ:
{"label": "bug_report", "summary": "Приложение не запускается и не загружается"}

Отзыв:
text: отлично приложение!!!
rating: 5
thumbs_up_count: 0
Ответ:
{"label": "rating", "summary": "Краткая положительная оценка без деталей"}

Отзыв:
text: прошу добавить темную тему
rating: 4
thumbs_up_count: 2
Ответ:
{"label": "feature_request", "summary": "Просьба добавить темную тему"}
"""
)


In [10]:
# Утилиты для инференса LLM

VALID_LABELS = {"bug_report", "feature_request", "user_experience", "rating"}


def format_review(row_or_text, rating=None, thumbs_up_count=None) -> str:
    """
    Приводит отзыв к тому формату, который описан в system_prompt.
    Можно передать либо строку, либо строку DataFrame.
    """
    if isinstance(row_or_text, pd.Series):
        text = str(row_or_text.get("text", ""))
        rating = row_or_text.get("rating", "")
        thumbs_up_count = row_or_text.get("thumbs_up_count", "")
    else:
        text = str(row_or_text)

    return (
        f"text: {text}\n"
        f"rating: {rating if rating is not None else ''}\n"
        f"thumbs_up_count: {thumbs_up_count if thumbs_up_count is not None else ''}"
    )


def prepare_messages(review_text: str, system_prompt_template: Template) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": system_prompt_template.render()},
        {"role": "user", "content": review_text},
    ]


def merge_system_into_user(messages):
    """
    Fallback для моделей/токенизаторов, которые не поддерживают отдельную role="system".
    Например, часть Mistral-шаблонов ожидает только user/assistant.
    В таком случае system_prompt аккуратно добавляется в первый user-message.
    """
    system_parts = []
    user_parts = []

    for msg in messages:
        role = msg.get("role", "")
        content = str(msg.get("content", ""))

        if role == "system":
            system_parts.append(content)
        elif role == "user":
            user_parts.append(content)
        else:
            # Для текущей задачи assistant-сообщений обычно нет, но оставим безопасную обработку.
            user_parts.append(content)

    system_text = "\n\n".join(system_parts).strip()
    user_text = "\n\n".join(user_parts).strip()

    if system_text:
        merged_content = (
            "Инструкция:\n"
            f"{system_text}\n\n"
            "Входные данные:\n"
            f"{user_text}"
        )
    else:
        merged_content = user_text

    return [{"role": "user", "content": merged_content}]


def apply_chat_template_safe(tokenizer, messages, enable_thinking: bool):
    """
    Вспомогательная функция, чтобы не дублировать параметры apply_chat_template.
    """
    kwargs = dict(
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    if enable_thinking:
        kwargs["enable_thinking"] = False

    return tokenizer.apply_chat_template(messages, **kwargs)


def build_chat_inputs(tokenizer, messages):
    """
    Универсальная подготовка входа для разных chat-моделей.

    У Phi-4-mini можно передавать enable_thinking=False, чтобы отключить reasoning.
    У Phi/Mistral/Llama такого аргумента может не быть, поэтому делаем fallback.
    У некоторых Mistral-шаблонов может не поддерживаться отдельная role="system",
    поэтому при ошибке system_prompt объединяется с user-сообщением.
    """
    # 1) Пробуем обычный chat-template с system/user.
    try:
        return apply_chat_template_safe(tokenizer, messages, enable_thinking=True)
    except TypeError:
        # Модель не знает аргумент enable_thinking.
        try:
            return apply_chat_template_safe(tokenizer, messages, enable_thinking=False)
        except Exception:
            pass
    except Exception:
        pass

    # 2) Fallback: объединяем system_prompt с user-сообщением.
    merged_messages = merge_system_into_user(messages)

    try:
        return apply_chat_template_safe(tokenizer, merged_messages, enable_thinking=True)
    except TypeError:
        return apply_chat_template_safe(tokenizer, merged_messages, enable_thinking=False)

def generate_response(model, tokenizer, messages, max_new_tokens=96):
    """
    Один вызов генерации для одного отзыва.
    Ответ должен быть коротким JSON, поэтому max_new_tokens оставляем маленьким.
    """
    model.eval()

    inputs = build_chat_inputs(tokenizer, messages)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            use_cache=True,
        )

    input_len = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    return response.strip()


def parse_llm_json(response: str) -> dict:
    """
    Пытается достать JSON из ответа модели.
    Если модель добавила лишний текст, вырезаем первый {...}.
    """
    raw_response = str(response)

    response = re.sub(r"<think>.*?</think>", "", raw_response, flags=re.DOTALL).strip()
    response = response.replace("```json", "").replace("```", "").strip()

    match = re.search(r"\{.*?\}", response, flags=re.DOTALL)
    if match:
        response = match.group(0)

    try:
        data = json.loads(response)
    except json.JSONDecodeError:
        return {
            "label": "parse_error",
            "summary": response[:120],
            "raw_response": raw_response,
        }

    label = str(data.get("label", "")).strip()
    summary = str(data.get("summary", "")).strip()

    if label not in VALID_LABELS:
        return {
            "label": "parse_error",
            "summary": summary[:120],
            "raw_response": raw_response,
        }

    # Жёстко ограничим summary на случай, если модель разговорилась
    summary = " ".join(summary.split()[:12])

    return {
        "label": label,
        "summary": summary,
        "raw_response": raw_response,
    }


In [11]:
def run_agent(row_or_text, model, tokenizer, system_prompt: Template, debug: bool = False):
    """
    Один отзыв -> один вызов model.generate -> dict с label и summary.
    Старый while True здесь не нужен, потому что tools/function calling не используются.
    """
    review_text = format_review(row_or_text)
    messages = prepare_messages(review_text, system_prompt)

    if debug:
        print("USER MESSAGE:")
        print(messages[-1]["content"])

    response = generate_response(
        model=model,
        tokenizer=tokenizer,
        messages=messages,
        max_new_tokens=96,
    )

    if debug:
        print("\nRAW MODEL RESPONSE:")
        print(response)

    return parse_llm_json(response)


In [12]:
# Батчевый инференс для тестовой выборки
# На тесте shuffle НЕ используется, чтобы сохранить исходный порядок строк.

def get_model_device(model):
    return next(model.parameters()).device


def render_prompt_text_for_generation(tokenizer, row, system_prompt_template: Template) -> str:
    review_text = format_review(row)
    messages = prepare_messages(review_text, system_prompt_template)

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        merged_messages = merge_system_into_user(messages)
        try:
            return tokenizer.apply_chat_template(
                merged_messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            return tokenizer.apply_chat_template(
                merged_messages,
                tokenize=False,
                add_generation_prompt=True,
            )


def run_agent_batch(
    batch_df,
    model,
    tokenizer,
    system_prompt: Template,
    max_new_tokens: int = 64,
    max_prompt_length: int = None,
):
    """
    Батчевый аналог run_agent.
    Возвращает список словарей: label, summary, raw_response.

    ВАЖНО:
    - Для генерации используем left padding.
    - Для truncation используем LEFT truncation, чтобы при длинном prompt не отрезать
      сам отзыв и assistant-generation prompt в конце. Иначе модель начинает
      продолжать few-shot примеры из system_prompt и может выдавать один класс
      для всех строк.
    """
    model.eval()

    if max_prompt_length is None:
        max_prompt_length = globals().get("MAX_SEQ_LENGTH", 768)

    old_padding_side = tokenizer.padding_side
    old_truncation_side = getattr(tokenizer, "truncation_side", "right")

    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    try:
        prompt_texts = [
            render_prompt_text_for_generation(tokenizer, row, system_prompt)
            for _, row in batch_df.iterrows()
        ]

        inputs = tokenizer(
            prompt_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_prompt_length,
            add_special_tokens=False,
        )

        input_len = inputs["input_ids"].shape[1]
        device = get_model_device(model)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                use_cache=True,
            )

        generated_ids = outputs[:, input_len:]
        responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        return [parse_llm_json(response.strip()) for response in responses]

    finally:
        tokenizer.padding_side = old_padding_side
        tokenizer.truncation_side = old_truncation_side


def debug_tokenized_generation_prompt(row, tokenizer, system_prompt_template: Template, max_prompt_length: int = None):
    """
    Диагностика: проверяет, что после tokenization/truncation в prompt остался
    текущий отзыв, а не только начало system_prompt с примерами.
    """
    if max_prompt_length is None:
        max_prompt_length = globals().get("MAX_SEQ_LENGTH", 768)

    old_padding_side = tokenizer.padding_side
    old_truncation_side = getattr(tokenizer, "truncation_side", "right")

    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    prompt_text = render_prompt_text_for_generation(tokenizer, row, system_prompt_template)
    inputs = tokenizer(
        [prompt_text],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_prompt_length,
        add_special_tokens=False,
    )
    decoded = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=False)

    tokenizer.padding_side = old_padding_side
    tokenizer.truncation_side = old_truncation_side

    print("Tokenized length:", int(inputs["input_ids"].shape[1]))
    print("Original review:")
    print(format_review(row))
    print("\nDecoded tail after truncation:")
    print(decoded[-1500:])


def run_batched_test_inference(
    test_df,
    model,
    tokenizer,
    system_prompt: Template,
    batch_size: int = 8,
    csv_path: Optional[Path] = None,
):
    """
    Батчевый инференс по test_df без перемешивания.
    Порядок строк сохраняется таким же, как в исходном test_df.
    """
    pred_rows = []

    for start in tqdm(range(0, len(test_df), batch_size), desc="Batched test inference"):
        batch_df = test_df.iloc[start:start + batch_size]
        preds = run_agent_batch(
            batch_df=batch_df,
            model=model,
            tokenizer=tokenizer,
            system_prompt=system_prompt,
        )

        for (idx, row), pred in zip(batch_df.iterrows(), preds):
            pred_rows.append({
                "id": idx,
                "text": row["text"],
                "rating": row.get("rating", None),
                "thumbs_up_count": row.get("thumbs_up_count", None),
                "label_gold": row.get("label_gold", None),
                "summary_gold": row.get("summary_gold", None),
                "label_pred": pred["label"],
                "summary_pred": pred["summary"],
                "raw_response": pred.get("raw_response", ""),
            })

    res_df = pd.DataFrame(pred_rows)

    print("Parse errors:", (res_df["label_pred"] == "parse_error").sum())

    valid_eval = res_df[res_df["label_pred"].isin(sorted(VALID_LABELS))].copy()

    print(classification_report(
        valid_eval["label_gold"],
        valid_eval["label_pred"],
        labels=["bug_report", "feature_request", "user_experience", "rating"],
        digits=4,
        zero_division=0,
    ))

    if csv_path is not None:
        res_df.to_csv(csv_path, index=False)
        print("Saved to:", csv_path)

    return res_df


# Fine-tuning и инференс Phi-4-mini-instruct

Ниже ноутбук работает в том же стиле, что и Qwen3-4B / Qwen2.5-7B версии:

1. загружается базовая модель `microsoft/Phi-4-mini-instruct`;
2. добавляется LoRA/QLoRA-адаптер;
3. модель дообучается на `train_data` без oversampling и без изменения распределения классов;
4. выполняется батчевый инференс по `test_data`;
5. предсказания сохраняются в CSV;
6. CSV можно заново прочитать и посчитать classification_report + summary-метрики.

Важное отличие Phi от Qwen: у Phi могут быть другие имена линейных слоёв (`qkv_proj`, `gate_up_proj`, `down_proj`), поэтому в ноутбуке есть автоматический подбор `TARGET_MODULES` для LoRA.


In [13]:
from transformers import BitsAndBytesConfig

# Лучше сохранять CSV и LoRA-адаптер прямо в папку проекта, чтобы они не потерялись после перезапуска Colab.
RESULTS_DIR = PROJECT_DIR / "LLM_LoRa_Finetune" /"llm_inference_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ADAPTERS_DIR = PROJECT_DIR / "LLM_LoRa_Finetune" / "llm_lora_adapters"
ADAPTERS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Режим дообучения
# ============================================================
# Для A100 / GPU с большим объёмом VRAM лучше начать с обычного LoRA:
# base model хранится в fp16/bf16, обучаются только LoRA-адаптеры.
#
#
# В QLoRA base model грузится в 4-bit NF4, а обучаются LoRA-адаптеры.
FINE_TUNE_MODE = "lora"   # "lora" или "qlora"

# Автоматически включаем 4-bit загрузку только для QLoRA.
LOAD_IN_4BIT = FINE_TUNE_MODE.lower() == "qlora"

# A100 обычно поддерживает bf16, поэтому для LoRA это лучший вариант.
# Если Colab/GPU ругается на bf16, поставь USE_BF16 = False.
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
TRAIN_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

bnb_config = None
if LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=TRAIN_DTYPE,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

def print_gpu_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU memory allocated: {allocated:.2f} GB")
        print(f"GPU memory reserved:  {reserved:.2f} GB")

print("RESULTS_DIR:", RESULTS_DIR)
print("ADAPTERS_DIR:", ADAPTERS_DIR)
print("FINE_TUNE_MODE:", FINE_TUNE_MODE)
print("LOAD_IN_4BIT:", LOAD_IN_4BIT)
print("TRAIN_DTYPE:", TRAIN_DTYPE)
print_gpu_memory()


RESULTS_DIR: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training/LLM_LoRa_Finetune/llm_inference_results
ADAPTERS_DIR: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training/LLM_LoRa_Finetune/llm_lora_adapters
FINE_TUNE_MODE: lora
LOAD_IN_4BIT: False
TRAIN_DTYPE: torch.bfloat16
GPU memory allocated: 0.00 GB
GPU memory reserved:  0.00 GB


## Phi-4-mini-instruct

In [14]:
# ==============================
# Модель: Phi-4-mini-instruct
# ==============================

model_name = "microsoft/Phi-4-mini-instruct"

# Для Phi ранее встречалась ошибка LossKwargs при trust_remote_code=True.
# Поэтому по умолчанию используем нативную поддержку Transformers без remote-code.
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=False,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Для обучения удобнее right padding, для батчевого generate ниже временно переключим на left padding.
tokenizer.padding_side = "right"

model_kwargs = {
    "device_map": "auto",
    "trust_remote_code": False,
    "torch_dtype": TRAIN_DTYPE,
}

if LOAD_IN_4BIT:
    model_kwargs["quantization_config"] = bnb_config

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    **model_kwargs,
)

model.eval()
print("Loaded:", model_name)
print_gpu_memory()


config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Loaded: microsoft/Phi-4-mini-instruct
GPU memory allocated: 7.15 GB
GPU memory reserved:  7.18 GB


# LoRA / QLoRA fine-tuning Phi-4-mini

Ниже добавлен supervised instruction fine-tuning под твою задачу:

- вход модели: `text`, `rating`, `thumbs_up_count`;
- целевой ответ модели: JSON с `label_gold` и `summary_gold`;
- обучение идёт по `cross-entropy` только на токенах ответа assistant;
- токены prompt маскируются через `labels = -100`, чтобы модель не училась переписывать инструкцию и сам отзыв.

Важно: `classification_report`, ROUGE-L и BERTScore не используются напрямую в `backward()`.  
Они считаются после генерации на validation/test как итоговые метрики качества.

## Почему выбраны именно эти слои

Для твоей задачи модель должна одновременно:
1. научиться выбирать один из четырёх классов;
2. научиться генерировать короткое summary;
3. стабильно соблюдать JSON-формат.

Поэтому в LoRA добавлены не только `q_proj`/`v_proj`, но и остальные attention-проекции (`k_proj`, `o_proj`) плюс MLP-проекции (`gate_proj`, `up_proj`, `down_proj`).  
Это чуть тяжелее, чем минимальный LoRA, но обычно лучше для instruction-following и генерации структурированного ответа.


In [16]:
# Подготовка SFT-датасета для instruction fine-tuning

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from peft import get_peft_model_state_dict, set_peft_model_state_dict


# ------------------------------
# Гиперпараметры обучения
# ------------------------------

SFT_VALID_SIZE = 0.15

# 768 обычно хватает для system_prompt + коротких отзывов и заметно экономит память.
# Если диагностика покажет обрезку assistant-target, вернуть 1024.
MAX_SEQ_LENGTH = 768

# Phi-4-mini меньше 7B-моделей, поэтому на A100 можно начинать с batch_size=8.
# Если будет OOM/нестабильность, уменьши TRAIN_BATCH_SIZE до 4.
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 6

# Для небольшой ручной разметки лучше начинать аккуратно.
# Слишком высокий LR может ухудшить instruction-following и привести к шаблонным ответам.
LEARNING_RATE = 2e-5 if FINE_TUNE_MODE == "lora" else 1e-4

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

# Ранняя остановка: не даём модели уходить в переобучение.
EARLY_STOPPING_PATIENCE = 2
MIN_DELTA = 1e-3

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.10


# Для Phi архитектура может использовать fused-проекции:
# attention: qkv_proj, o_proj
# MLP: gate_up_proj, down_proj
# Но в разных версиях Transformers имена могут отличаться, поэтому ниже есть автоподбор.
PHI_TARGET_MODULE_CANDIDATES = [
    ["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
    ["qkv_proj", "o_proj"],
    ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    ["q_proj", "k_proj", "v_proj", "o_proj"],
    ["q_proj", "v_proj"],
]


def get_available_linear_suffixes(model):
    """
    Собирает имена линейных модулей, к которым теоретически можно подключить LoRA.
    Работает и для обычных Linear, и для bitsandbytes Linear4bit/Linear8bitLt.
    """
    suffixes = set()

    for name, module in model.named_modules():
        if not name:
            continue

        cls_name = module.__class__.__name__.lower()
        is_linear_like = (
            isinstance(module, torch.nn.Linear)
            or "linear" in cls_name
            or "4bit" in cls_name
            or "8bit" in cls_name
        )

        if is_linear_like:
            suffix = name.split(".")[-1]
            if suffix not in {"lm_head"}:
                suffixes.add(suffix)

    return sorted(suffixes)


def resolve_target_modules(model, candidates):
    available = set(get_available_linear_suffixes(model))
    print("Available linear module suffixes:", sorted(available))

    for candidate in candidates:
        if all(module_name in available for module_name in candidate):
            print("Selected TARGET_MODULES:", candidate)
            return candidate

    raise ValueError(
        "Не удалось подобрать TARGET_MODULES для Phi. "
        f"Доступные linear suffixes: {sorted(available)}"
    )


TARGET_MODULES = resolve_target_modules(model, PHI_TARGET_MODULE_CANDIDATES)

ADAPTER_NAME = f"phi4_mini_instruct_{FINE_TUNE_MODE}"
ADAPTER_OUTPUT_DIR = ADAPTERS_DIR / ADAPTER_NAME
BEST_ADAPTER_OUTPUT_DIR = ADAPTERS_DIR / f"{ADAPTER_NAME}_best"

print("ADAPTER_OUTPUT_DIR:", ADAPTER_OUTPUT_DIR)
print("BEST_ADAPTER_OUTPUT_DIR:", BEST_ADAPTER_OUTPUT_DIR)
print("TARGET_MODULES:", TARGET_MODULES)


def build_target_json(row) -> str:
    """
    Целевой ответ для SFT.
    Именно этот JSON модель должна научиться генерировать.
    """
    label = str(row.get("label_gold", "")).strip()
    summary = str(row.get("summary_gold", "")).strip()

    return json.dumps(
        {
            "label": label,
            "summary": summary,
        },
        ensure_ascii=False,
    )


def merge_system_into_first_user_for_sft(messages):
    """
    Fallback для chat-template, если модель не поддерживает role="system".
    В отличие от merge_system_into_user для генерации, здесь сохраняем assistant-сообщение,
    чтобы SFT-таргет не превратился в часть user prompt.
    """
    system_text = "\n\n".join(
        str(m.get("content", "")) for m in messages if m.get("role") == "system"
    ).strip()

    merged = []
    system_inserted = False

    for msg in messages:
        role = msg.get("role", "")
        content = str(msg.get("content", ""))

        if role == "system":
            continue

        if role == "user" and system_text and not system_inserted:
            content = (
                "Инструкция:\n"
                f"{system_text}\n\n"
                "Входные данные:\n"
                f"{content}"
            )
            system_inserted = True

        merged.append({"role": role, "content": content})

    if system_text and not system_inserted:
        merged.insert(0, {"role": "user", "content": f"Инструкция:\n{system_text}"})

    return merged


def render_chat_text_for_sft(tokenizer, messages, add_generation_prompt: bool = False) -> str:
    """
    Рендер chat-template в текст.
    У Phi может не быть enable_thinking и/или отдельной role="system", поэтому есть fallback.
    """
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=False,
        )
    except TypeError:
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=add_generation_prompt,
            )
        except Exception:
            pass
    except Exception:
        pass

    merged_messages = merge_system_into_first_user_for_sft(messages)

    try:
        return tokenizer.apply_chat_template(
            merged_messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            merged_messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )


class ReviewSFTDataset(Dataset):
    """
    Dataset для causal LM fine-tuning.

    На входе:
    prompt = system + user(review)

    На выходе:
    assistant = {"label": label_gold, "summary": summary_gold}

    Критически важно:
    - prompt-токены имеют label = -100;
    - assistant-токены имеют реальные token ids;
    - при truncation мы резервируем место под assistant-ответ, чтобы loss не стал NaN.
    """

    def __init__(self, df, tokenizer, system_prompt_template, max_length=1024):
        self.df = df.reset_index(drop=True).copy()
        self.tokenizer = tokenizer
        self.system_prompt_template = system_prompt_template
        self.max_length = max_length

        self.df["text"] = self.df["text"].fillna("").astype(str)
        self.df["label_gold"] = self.df["label_gold"].fillna("").astype(str)
        self.df["summary_gold"] = self.df["summary_gold"].fillna("").astype(str)

        self.df = self.df[
            self.df["label_gold"].isin(sorted(VALID_LABELS))
            & (self.df["summary_gold"].str.strip() != "")
            & (self.df["text"].str.strip() != "")
        ].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        review_text = format_review(row)
        target_json = build_target_json(row)

        prompt_messages = prepare_messages(review_text, self.system_prompt_template)
        full_messages = prompt_messages + [
            {"role": "assistant", "content": target_json}
        ]

        prompt_text = render_chat_text_for_sft(
            self.tokenizer,
            prompt_messages,
            add_generation_prompt=True,
        )

        full_text = render_chat_text_for_sft(
            self.tokenizer,
            full_messages,
            add_generation_prompt=False,
        )

        # Лучше брать target_text прямо из chat-template, чтобы сохранить служебные токены окончания ответа.
        if full_text.startswith(prompt_text):
            target_text = full_text[len(prompt_text):]
        else:
            # Fallback на случай нестандартного chat-template.
            target_text = target_json
            if self.tokenizer.eos_token:
                target_text += self.tokenizer.eos_token

        prompt_ids = self.tokenizer(
            prompt_text,
            add_special_tokens=False,
        )["input_ids"]

        target_ids = self.tokenizer(
            target_text,
            add_special_tokens=False,
        )["input_ids"]

        # Защита от пустого target.
        if len(target_ids) == 0:
            target_ids = [self.tokenizer.eos_token_id]

        # Если target вдруг слишком длинный, режем target справа.
        # Для твоего JSON такого почти точно не будет.
        if len(target_ids) >= self.max_length:
            target_ids = target_ids[: self.max_length]
            prompt_ids = []
        else:
            # Резервируем место под target, а prompt при необходимости обрезаем.
            # Это устраняет главную причину NaN: batch без supervised-токенов.
            max_prompt_len = self.max_length - len(target_ids)
            if len(prompt_ids) > max_prompt_len:
                # Для коротких отзывов и MAX_SEQ_LENGTH=768 обычно не сработает.
                # Оставляем конец prompt, где находится сам отзыв и assistant-start.
                prompt_ids = prompt_ids[-max_prompt_len:]

        input_ids = prompt_ids + target_ids
        labels = [-100] * len(prompt_ids) + target_ids.copy()

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


class DataCollatorForCausalLMWithMaskedPrompt:
    """
    Паддинг батча:
    - input_ids паддим pad_token_id;
    - labels паддим -100, чтобы padding не участвовал в loss.
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    def __call__(self, features):
        input_ids = [f["input_ids"] for f in features]
        labels = [f["labels"] for f in features]

        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=self.pad_token_id,
        )

        labels = torch.nn.utils.rnn.pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100,
        )

        attention_mask = (input_ids != self.pad_token_id).long()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


# train/valid split только внутри train_data.
# test_data не трогаем: он остаётся финальным честным тестом.
sft_train_df, sft_valid_df = train_test_split(
    train_data,
    test_size=SFT_VALID_SIZE,
    random_state=RANDOM_SEED,
    stratify=train_data["label_gold"],
)

sft_train_df = sft_train_df.reset_index(drop=True)
sft_valid_df = sft_valid_df.reset_index(drop=True)

train_sft_dataset = ReviewSFTDataset(
    sft_train_df,
    tokenizer=tokenizer,
    system_prompt_template=system_prompt,
    max_length=MAX_SEQ_LENGTH,
)

valid_sft_dataset = ReviewSFTDataset(
    sft_valid_df,
    tokenizer=tokenizer,
    system_prompt_template=system_prompt,
    max_length=MAX_SEQ_LENGTH,
)

collator = DataCollatorForCausalLMWithMaskedPrompt(tokenizer)

train_loader = DataLoader(
    train_sft_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,      # На обучении перемешиваем для стохастичности.
    collate_fn=collator,
)

valid_loader = DataLoader(
    valid_sft_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,     # На validation порядок не важен, но не перемешиваем для воспроизводимости.
    collate_fn=collator,
)

print("SFT train examples:", len(train_sft_dataset))
print("SFT valid examples:", len(valid_sft_dataset))
print("Example target:", build_target_json(sft_train_df.iloc[0]))

# Диагностика: в каждом объекте должны быть supervised-токены, то есть labels != -100.
# Если здесь есть 0, loss может стать NaN.
sample_batch = next(iter(train_loader))
supervised_tokens = (sample_batch["labels"] != -100).sum(dim=1)
print("Supervised tokens in sample batch:", supervised_tokens.tolist())
print("Min supervised tokens:", int(supervised_tokens.min().item()))
print("Max sequence length in sample batch:", int(sample_batch["input_ids"].shape[1]))

if int(supervised_tokens.min().item()) == 0:
    raise ValueError(
        "В батче есть пример без supervised-токенов. "
        "Увеличь MAX_SEQ_LENGTH или проверь построение target_ids."
    )


Available linear module suffixes: ['down_proj', 'gate_up_proj', 'o_proj', 'qkv_proj']
Selected TARGET_MODULES: ['qkv_proj', 'o_proj', 'gate_up_proj', 'down_proj']
ADAPTER_OUTPUT_DIR: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training/LLM_LoRa_Finetune/llm_lora_adapters/phi4_mini_instruct_lora
BEST_ADAPTER_OUTPUT_DIR: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training/LLM_LoRa_Finetune/llm_lora_adapters/phi4_mini_instruct_lora_best
TARGET_MODULES: ['qkv_proj', 'o_proj', 'gate_up_proj', 'down_proj']
SFT train examples: 2306
SFT valid examples: 408
Example target: {"label": "user_experience", "summary": "Негативный опыт использования при блокировках и VPN"}
Supervised tokens in sample batch: [24, 22, 24, 24, 22, 21, 18, 25]
Min supervised tokens: 18
Max sequence length in sample batch: 599


In [17]:
# Подключение LoRA/QLoRA-адаптера к модели Phi

if FINE_TUNE_MODE == "qlora":
    # Нужно только для k-bit training.
    model = prepare_model_for_kbit_training(model)

# Отключаем cache во время обучения, особенно если включён gradient checkpointing.
model.config.use_cache = False

# Экономит память, но немного замедляет обучение.
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print_gpu_memory()


trainable params: 23,068,672 || all params: 3,859,090,432 || trainable%: 0.5978
GPU memory allocated: 7.23 GB
GPU memory reserved:  7.26 GB


In [18]:
# Обучение LoRA/QLoRA

def move_batch_to_device(batch, device):
    return {k: v.to(device) for k, v in batch.items()}


def evaluate_sft_loss(model, valid_loader):
    model.eval()
    losses = []
    device = get_model_device(model)

    with torch.no_grad():
        for batch in tqdm(valid_loader, desc="Validation loss"):
            batch = move_batch_to_device(batch, device)

            supervised_tokens = (batch["labels"] != -100).sum(dim=1)
            if int(supervised_tokens.min().item()) == 0:
                raise ValueError(
                    f"Validation batch contains empty labels: {supervised_tokens.tolist()}"
                )

            outputs = model(**batch)

            if not torch.isfinite(outputs.loss):
                raise FloatingPointError(
                    f"Non-finite validation loss: {outputs.loss.item()}. "
                    f"Supervised tokens: {supervised_tokens.tolist()}"
                )

            losses.append(outputs.loss.detach().float().item())

            # Чуть разгружаем память между validation-batch.
            del batch, outputs

    model.train()
    return float(np.mean(losses)) if losses else np.nan


num_update_steps_per_epoch = int(np.ceil(len(train_loader) / GRAD_ACCUM_STEPS))
num_training_steps = NUM_EPOCHS * num_update_steps_per_epoch
num_warmup_steps = int(num_training_steps * WARMUP_RATIO)

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

device = get_model_device(model)

train_history = []
global_step = 0

best_valid_loss = float("inf")
best_epoch = 0
best_adapter_state = None
patience_counter = 0

print("num_training_steps:", num_training_steps)
print("num_warmup_steps:", num_warmup_steps)
print("learning_rate:", LEARNING_RATE)
print("effective_batch_size:", TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)
print("early_stopping_patience:", EARLY_STOPPING_PATIENCE)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    seen_batches = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(progress, start=1):
        batch = move_batch_to_device(batch, device)

        supervised_tokens = (batch["labels"] != -100).sum(dim=1)
        if int(supervised_tokens.min().item()) == 0:
            raise ValueError(
                f"Train batch contains empty labels at step={step}: "
                f"{supervised_tokens.tolist()}"
            )

        outputs = model(**batch)

        if not torch.isfinite(outputs.loss):
            raise FloatingPointError(
                f"Non-finite train loss at epoch={epoch}, step={step}: "
                f"{outputs.loss.item()}. "
                f"Supervised tokens: {supervised_tokens.tolist()}. "
                f"Input shape: {tuple(batch['input_ids'].shape)}"
            )

        loss = outputs.loss / GRAD_ACCUM_STEPS
        loss.backward()

        running_loss += outputs.loss.detach().float().item()
        seen_batches += 1

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1

            avg_loss = running_loss / max(seen_batches, 1)
            progress.set_postfix({
                "train_loss": f"{avg_loss:.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}",
            })

        del batch, outputs, loss

    train_loss = running_loss / max(seen_batches, 1)
    valid_loss = evaluate_sft_loss(model, valid_loader)

    improved = valid_loss < (best_valid_loss - MIN_DELTA)

    if improved:
        best_valid_loss = valid_loss
        best_epoch = epoch
        patience_counter = 0

        # Сохраняем лучший adapter на диск и в RAM/CPU, чтобы после обучения
        # не использовать переобученную последнюю эпоху.
        BEST_ADAPTER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(BEST_ADAPTER_OUTPUT_DIR)
        tokenizer.save_pretrained(BEST_ADAPTER_OUTPUT_DIR)

        best_adapter_state = {
            k: v.detach().cpu().clone()
            for k, v in get_peft_model_state_dict(model).items()
        }
        best_marker = " <-- best"
    else:
        patience_counter += 1
        best_marker = ""

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": valid_loss,
        "best_valid_loss": best_valid_loss,
        "is_best": improved,
        "patience_counter": patience_counter,
    }

    train_history.append(row)

    print(
        f"Epoch {epoch}: "
        f"train_loss={train_loss:.4f}, "
        f"valid_loss={valid_loss:.4f}, "
        f"best_valid_loss={best_valid_loss:.4f}, "
        f"patience={patience_counter}/{EARLY_STOPPING_PATIENCE}"
        f"{best_marker}"
    )

    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}")
        break

# Возвращаем в память лучший adapter, а не последнюю переобученную эпоху.
if best_adapter_state is not None:
    set_peft_model_state_dict(model, best_adapter_state)
    print(f"Restored best adapter from epoch {best_epoch}, valid_loss={best_valid_loss:.4f}")
else:
    print("Warning: best_adapter_state is None; current adapter was not restored.")

train_history_df = pd.DataFrame(train_history)
train_history_df


num_training_steps: 222
num_warmup_steps: 11
learning_rate: 2e-05
effective_batch_size: 64
early_stopping_patience: 2


Epoch 1/6:   0%|          | 0/289 [00:00<?, ?it/s]

Validation loss:   0%|          | 0/51 [00:00<?, ?it/s]

Epoch 1: train_loss=0.9025, valid_loss=0.5011, best_valid_loss=0.5011, patience=0/2 <-- best


Epoch 2/6:   0%|          | 0/289 [00:00<?, ?it/s]

...

Epoch 5: train_loss=0.2278, valid_loss=0.2253, best_valid_loss=0.2253, patience=0/2 <-- best


Epoch 6/6:   0%|          | 0/289 [00:00<?, ?it/s]

Validation loss:   0%|          | 0/51 [00:00<?, ?it/s]

Epoch 6: train_loss=0.2173, valid_loss=0.2210, best_valid_loss=0.2210, patience=0/2 <-- best
Restored best adapter from epoch 6, valid_loss=0.2210


,epoch,train_loss,valid_loss,best_valid_loss,is_best,patience_counter
0,1,0.902471,0.501072,0.501072,True,0
1,2,0.421719,0.346145,0.346145,True,0
2,3,0.307171,0.270012,0.270012,True,0
3,4,0.252553,0.239017,0.239017,True,0
4,5,0.227755,0.225264,0.225264,True,0
5,6,0.217319,0.221007,0.221007,True,0


In [19]:
ADAPTER_OUTPUT_DIR

PosixPath('/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training/LLM_LoRa_Finetune/llm_lora_adapters/phi4_mini_instruct_lora')

In [20]:
# Сохранение LoRA/QLoRA-адаптера

# Сохраняем текущий adapter как final.
# ВАЖНО: после ячейки обучения в памяти уже восстановлен best adapter.
# Поэтому для честного инференса лучше использовать BEST_ADAPTER_OUTPUT_DIR или текущую модель после restore.
ADAPTER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(ADAPTER_OUTPUT_DIR)
tokenizer.save_pretrained(ADAPTER_OUTPUT_DIR)

print("Final/current adapter saved to:", ADAPTER_OUTPUT_DIR)
print("Best adapter saved to:", BEST_ADAPTER_OUTPUT_DIR)
print_gpu_memory()


Final/current adapter saved to: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training/LLM_LoRa_Finetune/llm_lora_adapters/phi4_mini_instruct_lora
Best adapter saved to: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training/LLM_LoRa_Finetune/llm_lora_adapters/phi4_mini_instruct_lora_best
GPU memory allocated: 7.42 GB
GPU memory reserved:  79.27 GB


## Опционально: загрузка сохранённого адаптера после перезапуска

Если ты уже обучил адаптер, сохранил его и перезапустил Colab, можно не обучать заново.  
В таком случае выполни загрузку base model выше, пропусти ячейки обучения и выполни ячейку ниже для подключения сохранённого адаптера.


In [ ]:
# Опциональная загрузка уже сохранённого LoRA/QLoRA adapter после перезапуска
# Выполняй эту ячейку только если adapter уже был обучен и сохранён.
# Для инференса лучше загружать BEST_ADAPTER_OUTPUT_DIR, а не последнюю эпоху.

# from peft import PeftModel
#
# ADAPTER_TO_LOAD = BEST_ADAPTER_OUTPUT_DIR
# model = PeftModel.from_pretrained(model, ADAPTER_TO_LOAD)
# model.eval()
# model.config.use_cache = True
#
# print("Loaded adapter:", ADAPTER_TO_LOAD)
# print_gpu_memory()


In [21]:
# Батчевый инференс по всей тестовой выборке + classification_report
# Модель: Phi-4-mini-instruct + лучший LoRA/QLoRA adapter

INFERENCE_BATCH_SIZE = 8

# После обучения включаем cache обратно для более быстрого generate.
# Gradient checkpointing нужен только для обучения и может мешать/замедлять generate.
try:
    model.gradient_checkpointing_disable()
except Exception:
    pass

model.config.use_cache = True
model.eval()

csv_path = RESULTS_DIR / f"{ADAPTER_NAME}_best_predictions.csv"

# Диагностика перед полным тестом:
# проверяем, что truncation не отрезает текущий отзыв и assistant prompt.
debug_tokenized_generation_prompt(
    row=test_data.iloc[0],
    tokenizer=tokenizer,
    system_prompt_template=system_prompt,
    max_prompt_length=MAX_SEQ_LENGTH,
)

res_phi4_mini_instruct_finetuned = run_batched_test_inference(
    test_df=test_data,
    model=model,
    tokenizer=tokenizer,
    system_prompt=system_prompt,
    batch_size=INFERENCE_BATCH_SIZE,
    csv_path=csv_path,
)

res_df = res_phi4_mini_instruct_finetuned.copy()
res_df.head()


Tokenized length: 497
Original review:
text: с каждым новым обновлением всё хуже, даже сообщения не доходят до меня, даже посмотреть их не могу
rating: 1
thumbs_up_count: 0

Decoded tail after truncation:
rating

Правила разметки:
- bug_report: пользователь сообщает об ошибке, сбое, зависании, некорректной работе, потере данных.
- feature_request: пользователь просит добавить или изменить функцию. Если одновременно есть явная ошибка и просьба, выбери bug_report.
- user_experience: пользователь описывает опыт использования, удобство, неудобство, плюсы/минусы без явного бага и без просьбы новой функции.
- rating: короткая общая оценка без полезной конкретики.

Приоритет при смешанных случаях:
bug_report > feature_request > user_experience > rating

2. Сформировать summary:
- одна короткая фраза на русском;
- не более 12 слов;
- без выдуманных деталей;
- только главная мысль из отзыва.

Верни только валидный JSON без markdown и пояснений:
{"label": "bug_report|feature_request|user_experie

Batched test inference:   0%|          | 0/60 [00:00<?, ?it/s]

Parse errors: 0
                 precision    recall  f1-score   support

     bug_report     0.7419    0.7360    0.7390       125
feature_request     0.7778    0.2692    0.4000        26
user_experience     0.7510    0.8465    0.7959       228
         rating     0.8889    0.7921    0.8377       101

       accuracy                         0.7750       480
      macro avg     0.7899    0.6610    0.6931       480
   weighted avg     0.7791    0.7750    0.7684       480

Saved to: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training/LLM_LoRa_Finetune/llm_inference_results/phi4_mini_instruct_lora_best_predictions.csv


,id,text,rating,thumbs_up_count,label_gold,summary_gold,label_pred,summary_pred,raw_response
0,0,"с каждым новым обновлением всё хуже, даже сооб...",1,0,user_experience,Проблемы с чатами и сообщениями,bug_report,Проблемы с сообщениями,"{""label"": ""bug_report"", ""summary"": ""Проблемы с..."
1,1,"приложение супер, только почему-то после того ...",5,0,user_experience,Приложение работает некорректно,bug_report,Приложение работает некорректно,"{""label"": ""bug_report"", ""summary"": ""Приложение..."
2,2,при включении музыки постоянно виснит и вылета...,1,1,bug_report,Приложение лагает и тормозит,bug_report,Проблемы с музыкой,"{""label"": ""bug_report"", ""summary"": ""Проблемы с..."
3,3,после обновления приложения не работает не мог...,1,0,bug_report,Приложение работает некорректно,bug_report,Приложение работает некорректно,"{""label"": ""bug_report"", ""summary"": ""Приложение..."
4,4,очень неудобное приложение. проблем внутри куч...,1,0,feature_request,Предложение по улучшению приложения,user_experience,Проблемы с оплатой,"{""label"": ""user_experience"", ""summary"": ""Пробл..."


In [22]:
res_df[res_df["label_pred"] == "parse_error"]

,id,text,rating,thumbs_up_count,label_gold,summary_gold,label_pred,summary_pred,raw_response


In [ ]:
# # Удаляем модель из памяти после инференса: Phi-4-mini-instruct

# del model
# del tokenizer

# gc.collect()

# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
#     torch.cuda.ipc_collect()

# print("Model deleted:", "Phi-4-mini-instruct")
# print_gpu_memory()


In [23]:
# Список сохранённых файлов с предсказаниями

prediction_files = {
    "Phi-4-mini-instruct-LoRA": RESULTS_DIR / "phi4_mini_instruct_lora_best_predictions.csv",
    # "Phi-4-mini-instruct-QLoRA": RESULTS_DIR / "phi4_mini_instruct_qlora_best_predictions.csv",
}

# Оставим только реально существующие файлы, чтобы ячейки сравнения не падали.
prediction_files = {
    name: path
    for name, path in prediction_files.items()
    if path.exists()
}

prediction_files


{'Phi-4-mini-instruct-LoRA': PosixPath('/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training/LLM_LoRa_Finetune/llm_inference_results/phi4_mini_instruct_lora_best_predictions.csv')}

In [24]:
import re
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from bert_score import score as bert_score


def normalize_text(text):
    """
    Нормализация текста summary:
    - приводим к строке;
    - убираем лишние пробелы;
    - приводим к нижнему регистру;
    - заменяем ё на е.
    """
    if pd.isna(text):
        return ""

    text = str(text).strip().lower()
    text = text.replace("ё", "е")
    text = re.sub(r"\s+", " ", text)

    return text


def tokenize_ru(text):
    """
    Простая токенизация для русскоязычного текста.
    Берем русские/английские слова и числа.
    """
    text = normalize_text(text)
    return re.findall(r"[а-яa-z0-9]+", text)


def lcs_length(tokens_a, tokens_b):
    """
    Длина наибольшей общей подпоследовательности.
    Нужна для ROUGE-L.
    """
    n = len(tokens_a)
    m = len(tokens_b)

    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n):
        for j in range(m):
            if tokens_a[i] == tokens_b[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[n][m]


def rouge_l_score(reference, prediction):
    """
    Считает ROUGE-L Precision, Recall и F1 для одной пары:
    reference = summary_gold
    prediction = summary_pred
    """
    ref_tokens = tokenize_ru(reference)
    pred_tokens = tokenize_ru(prediction)

    if len(ref_tokens) == 0 and len(pred_tokens) == 0:
        return 1.0, 1.0, 1.0

    if len(ref_tokens) == 0 or len(pred_tokens) == 0:
        return 0.0, 0.0, 0.0

    lcs = lcs_length(ref_tokens, pred_tokens)

    precision = lcs / len(pred_tokens)
    recall = lcs / len(ref_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1


def evaluate_summaries(
    df,
    gold_col="summary_gold",
    pred_col="summary_pred",
    label_col=None,
    bert_model="xlm-roberta-base",
    batch_size=16
):
    """
    Оценивает качество summary по:
    - ROUGE-L;
    - BERTScore.

    Возвращает:
    - detailed_df: исходный датафрейм + метрики для каждой строки;
    - overall_metrics: общие средние метрики;
    - by_label_metrics: метрики по классам, если передан label_col.
    """

    detailed_df = df.copy()

    detailed_df[gold_col] = detailed_df[gold_col].fillna("").astype(str)
    detailed_df[pred_col] = detailed_df[pred_col].fillna("").astype(str)

    # ---------- ROUGE-L ----------
    rouge_p_list = []
    rouge_r_list = []
    rouge_f1_list = []

    for gold, pred in tqdm(
        zip(detailed_df[gold_col], detailed_df[pred_col]),
        total=len(detailed_df),
        desc="Calculating ROUGE-L"
    ):
        p, r, f1 = rouge_l_score(gold, pred)
        rouge_p_list.append(p)
        rouge_r_list.append(r)
        rouge_f1_list.append(f1)

    detailed_df["rougeL_precision"] = rouge_p_list
    detailed_df["rougeL_recall"] = rouge_r_list
    detailed_df["rougeL_f1"] = rouge_f1_list

    # ---------- BERTScore ----------
    gold_texts = detailed_df[gold_col].apply(normalize_text).tolist()
    pred_texts = detailed_df[pred_col].apply(normalize_text).tolist()

    valid_mask = [
        len(gold.strip()) > 0 and len(pred.strip()) > 0
        for gold, pred in zip(gold_texts, pred_texts)
    ]

    detailed_df["bertscore_precision"] = 0.0
    detailed_df["bertscore_recall"] = 0.0
    detailed_df["bertscore_f1"] = 0.0

    valid_gold = [gold for gold, is_valid in zip(gold_texts, valid_mask) if is_valid]
    valid_pred = [pred for pred, is_valid in zip(pred_texts, valid_mask) if is_valid]

    if len(valid_gold) > 0:
        device = "cuda" if torch.cuda.is_available() else "cpu"

        P, R, F1 = bert_score(
            cands=valid_pred,
            refs=valid_gold,
            model_type=bert_model,
            batch_size=batch_size,
            device=device,
            verbose=True
        )

        valid_indices = detailed_df.index[valid_mask]

        detailed_df.loc[valid_indices, "bertscore_precision"] = P.cpu().numpy()
        detailed_df.loc[valid_indices, "bertscore_recall"] = R.cpu().numpy()
        detailed_df.loc[valid_indices, "bertscore_f1"] = F1.cpu().numpy()

    # ---------- Общие метрики ----------
    metric_cols = [
        "rougeL_precision",
        "rougeL_recall",
        "rougeL_f1",
        "bertscore_precision",
        "bertscore_recall",
        "bertscore_f1"
    ]

    overall_metrics = detailed_df[metric_cols].mean().to_frame("mean_value")

    # ---------- Метрики по классам ----------
    by_label_metrics = None

    if label_col is not None and label_col in detailed_df.columns:
        by_label_metrics = (
            detailed_df
            .groupby(label_col)[metric_cols]
            .mean()
            .sort_values("bertscore_f1", ascending=False)
        )

    return detailed_df, overall_metrics, by_label_metrics

# Сравнение метрик по сохранённым CSV

Эта ячейка читает сохранённые файлы предсказаний и собирает одну таблицу по классификации и summary-метрикам.

In [25]:
# Пример: открыть конкретный CSV и посмотреть classification_report детально

model_to_inspect = list(prediction_files.keys())[0]
inspect_df = pd.read_csv(prediction_files[model_to_inspect])

valid_eval = inspect_df[inspect_df["label_pred"].isin(sorted(VALID_LABELS))].copy()

print("Model:", model_to_inspect)
print("Parse errors:", (inspect_df["label_pred"] == "parse_error").sum())

print(classification_report(
    valid_eval["label_gold"],
    valid_eval["label_pred"],
    labels=["bug_report", "feature_request", "user_experience", "rating"],
    digits=4,
    zero_division=0,
))

inspect_df.head()


Model: Phi-4-mini-instruct-LoRA
Parse errors: 0
                 precision    recall  f1-score   support

     bug_report     0.7419    0.7360    0.7390       125
feature_request     0.7778    0.2692    0.4000        26
user_experience     0.7510    0.8465    0.7959       228
         rating     0.8889    0.7921    0.8377       101

       accuracy                         0.7750       480
      macro avg     0.7899    0.6610    0.6931       480
   weighted avg     0.7791    0.7750    0.7684       480



,id,text,rating,thumbs_up_count,label_gold,summary_gold,label_pred,summary_pred,raw_response
0,0,"с каждым новым обновлением всё хуже, даже сооб...",1,0,user_experience,Проблемы с чатами и сообщениями,bug_report,Проблемы с сообщениями,"{""label"": ""bug_report"", ""summary"": ""Проблемы с..."
1,1,"приложение супер, только почему-то после того ...",5,0,user_experience,Приложение работает некорректно,bug_report,Приложение работает некорректно,"{""label"": ""bug_report"", ""summary"": ""Приложение..."
2,2,при включении музыки постоянно виснит и вылета...,1,1,bug_report,Приложение лагает и тормозит,bug_report,Проблемы с музыкой,"{""label"": ""bug_report"", ""summary"": ""Проблемы с..."
3,3,после обновления приложения не работает не мог...,1,0,bug_report,Приложение работает некорректно,bug_report,Приложение работает некорректно,"{""label"": ""bug_report"", ""summary"": ""Приложение..."
4,4,очень неудобное приложение. проблем внутри куч...,1,0,feature_request,Предложение по улучшению приложения,user_experience,Проблемы с оплатой,"{""label"": ""user_experience"", ""summary"": ""Пробл..."


In [26]:
summary_eval_df, summary_overall_metrics, _ = evaluate_summaries(
    inspect_df,
    gold_col="summary_gold",
    pred_col="summary_pred",
    bert_model="xlm-roberta-base",
    batch_size=16
)

summary_overall_metrics

Calculating ROUGE-L:   0%|          | 0/480 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

  0%|          | 0/13 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/30 [00:00<?, ?it/s]

done in 0.49 seconds, 982.36 sentences/sec


,mean_value
rougeL_precision,0.445637
rougeL_recall,0.440537
rougeL_f1,0.439438
bertscore_precision,0.902130
bertscore_recall,0.898452
bertscore_f1,0.900171


In [27]:
summary_eval_df[
    ["text", "summary_gold", "summary_pred", "rougeL_f1", "bertscore_f1"]
].sort_values("bertscore_f1").head(20)

,text,summary_gold,summary_pred,rougeL_f1,bertscore_f1
479,"ребят, а где теперь калибровать компас?",где калибровать компас,Проблемы с навигацией,0.0,0.734277
124,Приложение просто замечательнО! Но есть не бол...,Добавить галочки прочтения сообщений,Проблемы с чатами и сообщениями,0.0,0.770052
266,"Обновила вк, теперь когда входишь в сообщения ...",Приложение вылетает,Проблемы с сообщениями,0.0,0.779555
13,"Плохо, очень плохо вконтакте -постоянные баги ...",Приложение вылетает,Проблемы с загрузкой и загрузкой данных,0.0,0.780033
133,добавьте пожалуйста кнопку для грузчика есть п...,Добавить отметку пропуска на МКАД,Проблемы с навигацией,0.0,0.782650
74,сеть не работает,Приложение работает некорректно,Проблемы с сетью,0.0,0.784674
451,что за проблемы со входом????в день по два раз...,Приложение вылетает,Проблемы с входом и авторизацией,0.0,0.784681
96,"Вроде оф приложения, но такое не удобное. Ладн...",Добавить настройки сообщений,Приложение работает некорректно,0.0,0.786664
205,раньше была проблема. Зачем убрали возможность...,Неудобное использование приложения,Проблемы с навигацией,0.0,0.786937
349,Куча рекламы. Сделай те уже возможность пересы...,Добавить отключение рекламы,Недовольство модерацией и поддержкой,0.0,0.791102


In [28]:
summary_eval_df[
    ["text", "summary_gold", "summary_pred", "rougeL_f1", "bertscore_f1"]
].sort_values("bertscore_f1", ascending=False).head(20)

,text,summary_gold,summary_pred,rougeL_f1,bertscore_f1
240,в этом приложение всё есть пользуюсь только им.,Положительный опыт использования приложения,Положительный опыт использования приложения,1.0,1.0
229,"Отличное приложение, смешно с женщин которые р...",Положительный опыт использования приложения,Положительный опыт использования приложения,1.0,1.0
253,"авито работает хорошо,всегда можно дозвониться.",Положительный опыт использования приложения,Положительный опыт использования приложения,1.0,1.0
337,Авито расширяет кругозор от бренда до умельцев...,Положительный опыт использования приложения,Положительный опыт использования приложения,1.0,1.0
172,телеграм сила Макс могила,Положительный опыт использования приложения,Положительный опыт использования приложения,1.0,1.0
459,чтоб без вас делали не знаю вы лучшие.,Положительный опыт использования приложения,Положительный опыт использования приложения,1.0,1.0
27,Беспонтовые запросы у авито с паспортом и виде...,Негативный опыт использования приложения,Негативный опыт использования приложения,1.0,1.0
417,Наконец-то теперь можно заказать товар и оплат...,Положительный опыт использования приложения,Положительный опыт использования приложения,1.0,1.0
57,Улучшается с каждым днём. Мне нравится и я пол...,Положительный опыт использования приложения,Положительный опыт использования приложения,1.0,1.0
357,Спасибо за оперативную техническую поддержку!!!,Положительный опыт использования приложения,Положительный опыт использования приложения,1.0,1.0


In [29]:
summary_eval_df[
    ["text", "summary_gold", "summary_pred", "rougeL_f1", "bertscore_f1"]
][summary_eval_df["rougeL_f1"].between(0.6, 0.8)]

,text,summary_gold,summary_pred,rougeL_f1,bertscore_f1
0,"с каждым новым обновлением всё хуже, даже сооб...",Проблемы с чатами и сообщениями,Проблемы с сообщениями,0.750000,0.953227
11,"Не загружает транспорт, еле еле строит маршрут...",Проблемы с маршрутами и навигацией,Проблемы с маршрутизацией и геолокацией,0.600000,0.904958
39,Приложение после обновления разочаровало из-за...,вернуть выбор способа передвижения,Добавить выбор способа передвижения,0.750000,0.959413
58,"удалили вк админ, чтоб владельцы сообществ вер...",Проблемы с сообщениями,Проблемы с сообщениями и уведомлениями,0.750000,0.946231
68,тормоза одним словом,Приложение лагает и тормозит,Приложение тормозит,0.666667,0.891963
121,сапибо за вежливость,Положительный опыт использования сервиса такси,Положительный опыт использования приложения,0.666667,0.933193
153,вежливы приятный молодой человек.,Положительный опыт использования сервиса такси,Положительный опыт использования приложения,0.666667,0.933193
158,Добрый и эмпатичный человек. Предложил свою по...,Положительный опыт использования сервиса такси,Положительный опыт использования приложения,0.666667,0.933193
177,"Быстро довёз,",Положительный опыт использования сервиса такси,Положительный опыт использования приложения,0.666667,0.933193
183,"Если бы не блокировка вк для музыки, то бы не ...",Проблемы с загрузкой данных,Проблемы с загрузкой музыки,0.750000,0.953205
